In [14]:
import pandas as pd
import numpy as np
import re


# Parameters 
chromosome = "18"
sigma = 10000
enhancer_path = "/pool01/projects/abante_lab/ao_prediction_enrollhd_2024/genes/genehancer.txt"
predictions_path = "/pool01/projects/abante_lab/genomic_llms/borzoi/test_outputs/sed_all_chr/chr18.txt"

# Load Enhancer File 
enh = pd.read_csv(enhancer_path, sep="\t", dtype=str)  # uses header from file

enh.columns = [
    "chrom", "source", "feature", "start", "end", "score",
    "strand", "phase", "connected_genes", "genehancer_id", "gene_score_dict"
]

# Convert positions to integers
enh["start"] = enh["start"].astype(int)
enh["end"] = enh["end"].astype(int)

# Normalize chromosome naming (remove 'chr')
enh["chrom"] = enh["chrom"].str.replace("chr", "", regex=False)

# Filter to target chromosome only 
enh = enh[enh["chrom"] == chromosome]

print(enh.shape)
print(enh.head())


(5313, 11)
    chrom      source   feature     start       end score strand phase  \
1      18  GeneHancer  Enhancer  48592690  48592839  0.96      .     .   
72     18  GeneHancer  Enhancer  77251637  77251927  0.54      .     .   
76     18  GeneHancer  Enhancer  76951415  76951992  0.77      .     .   
111    18  GeneHancer  Enhancer  45723601  45725000  0.25      .     .   
138    18  GeneHancer  Enhancer  48588935  48591239  1.09      .     .   

                                       connected_genes genehancer_id  \
1    genehancer_id=GH18F048592;connected_gene=ENSG0...   GH18F048592   
72   genehancer_id=GH18F077251;connected_gene=GALR1...   GH18F077251   
76   genehancer_id=GH18F076951;connected_gene=RNU6-...   GH18F076951   
111  genehancer_id=GH18F045723;connected_gene=SLC14...   GH18F045723   
138  genehancer_id=GH18F048588;connected_gene=RPL17...   GH18F048588   

                                       gene_score_dict  
1    {'ENSG00000278983': 0.29, 'ENSG00000279629': 0...

In [15]:

# Extract ENSG IDs Only 
def extract_ensembl_genes(row):
    raw = row.get("connected_genes", "")
    gene_ids = [
        part.split("=")[1]
        for part in raw.split(";")
        if part.startswith("connected_gene=") and re.match(r"ENSG\d+", part.split("=")[1])
    ]
    return gene_ids

enh["ensembl_gene_ids"] = enh.apply(extract_ensembl_genes, axis=1)

# Debug Print 
print("Sample ENSG gene IDs extracted:")
print(enh[["genehancer_id", "ensembl_gene_ids"]].head(10))



Sample ENSG gene IDs extracted:
    genehancer_id                    ensembl_gene_ids
1     GH18F048592  [ENSG00000278983, ENSG00000279629]
72    GH18F077251                   [ENSG00000264015]
76    GH18F076951                                  []
111   GH18F045723                                  []
138   GH18F048588  [ENSG00000278983, ENSG00000279629]
178   GH18F008614                   [ENSG00000266708]
179   GH18F010246                                  []
182   GH18F010766                                  []
234   GH18F058201                   [ENSG00000267504]
258   GH18F023340                   [ENSG00000266495]


In [24]:

# Load Predictions File 
preds = pd.read_csv(predictions_path, sep="\t")
preds = preds.copy()
preds = preds.head(200)
print(preds)

# Extract SNP position and chromosome
snp_parts = preds["snp"].str.split("_", expand=True)
preds["chrom"] = snp_parts[0]
preds["pos"] = snp_parts[1].astype(int)



                 snp             gene alt_allele            tissue    logSED
0     18_9806921_A_G  ENSG00000168461          G  RNA-seq: Putamen  0.000712
1     18_9806921_A_G  ENSG00000168461          G  RNA-seq: Caudate  0.000730
2     18_9806921_A_G  ENSG00000285653          G  RNA-seq: Putamen  0.000267
3     18_9806921_A_G  ENSG00000285653          G  RNA-seq: Caudate  0.000026
4     18_9806921_A_G  ENSG00000168454          G  RNA-seq: Putamen  0.000246
..               ...              ...        ...               ...       ...
195  18_13445171_C_G  ENSG00000267136          G  RNA-seq: Caudate  0.000035
196  18_13445171_C_G  ENSG00000272746          G  RNA-seq: Putamen -0.000320
197  18_13445171_C_G  ENSG00000272746          G  RNA-seq: Caudate  0.000019
198  18_46468251_T_C  ENSG00000134030          C  RNA-seq: Putamen -0.000452
199  18_46468251_T_C  ENSG00000134030          C  RNA-seq: Caudate -0.000215

[200 rows x 5 columns]


In [25]:

# --- Match and Weight ---
def compute_weighted_logsed(row):
    gene = row["gene"]
    pos = row["pos"]

    gene_enhancers = enh[enh["gene"] == gene]
    if gene_enhancers.empty:
        return 0.0

    centers = (gene_enhancers["start"] + gene_enhancers["end"]) / 2
    dists = np.abs(centers - pos)
    weights = np.exp(- (dists ** 2) / (2 * sigma ** 2))
    max_weight = weights.max()

    return row["logSED"] * max_weight



In [26]:
def compute_weighted_logsed(row):
    gene = row["gene"]
    pos = row["pos"]
    #print(enh["ensembl_gene_ids"])
    gene_enhancers = enh[enh["ensembl_gene_ids"].apply(lambda gene_list: gene in gene_list)]
    print(gene_enhancers.shape)
    if gene_enhancers.empty:
        return 0.0
    
    centers = (gene_enhancers["start"] + gene_enhancers["end"]) / 2
    dists = np.abs(centers - pos)
    weights = np.exp(- (dists ** 2) / (2 * sigma ** 2))
    max_weight = weights.max()
    #print(f"Gene: {gene}, Position: {pos}, Centers: {centers.tolist()}, Distances: {dists.tolist()}, Weights: {weights.tolist()}, Max Weight: {max_weight}")

    return row["logSED"] * max_weight

preds["weighted_logSED"] = preds.apply(compute_weighted_logsed, axis=1)




(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(6, 12)
(6, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(12, 12)
(12, 12)
(14, 12)
(14, 12)
(0, 12)
(0, 12)
(6, 12)
(6, 12)
(0, 12)
(0, 12)
(3, 12)
(3, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(10, 12)
(10, 12)
(0, 12)
(0, 12)
(6, 12)
(6, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(13, 12)
(13, 12)
(1, 12)
(1, 12)
(12, 12)
(12, 12)
(0, 12)
(0, 12)
(10, 12)
(10, 12)
(1, 12)
(1, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(18, 12)
(18, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(5, 12)
(5, 12)
(0, 12)
(0, 12)
(3, 12)
(3, 12)
(0, 12)
(0, 12)
(5, 12)
(5, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(2, 12)
(2, 12)
(17, 12)
(17, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(12, 12)
(12, 12)
(14, 12)
(14, 12)
(2, 12)
(2, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 12)
(0, 

In [28]:
# Output 
print(preds[["snp", "gene", "logSED", "weighted_logSED"]])
# preds.to_csv("chr18_weighted_predictions.tsv", sep="\t", index=False)

                 snp             gene    logSED  weighted_logSED
0     18_9806921_A_G  ENSG00000168461  0.000712     0.000000e+00
1     18_9806921_A_G  ENSG00000168461  0.000730     0.000000e+00
2     18_9806921_A_G  ENSG00000285653  0.000267     0.000000e+00
3     18_9806921_A_G  ENSG00000285653  0.000026     0.000000e+00
4     18_9806921_A_G  ENSG00000168454  0.000246     0.000000e+00
..               ...              ...       ...              ...
195  18_13445171_C_G  ENSG00000267136  0.000035     1.123190e-14
196  18_13445171_C_G  ENSG00000272746 -0.000320    -4.599569e-11
197  18_13445171_C_G  ENSG00000272746  0.000019     2.678775e-12
198  18_46468251_T_C  ENSG00000134030 -0.000452     0.000000e+00
199  18_46468251_T_C  ENSG00000134030 -0.000215     0.000000e+00

[200 rows x 4 columns]
